# 11 — Retrieval

Modules 08 and 09 answered Helena from a **database**. Module 02 answered Amsterdam from a **CSV**. Those answers have an address: a table, a row, a function.

A return window does not. It lives in a paragraph in `data/corpus/`. You cannot `SELECT` it. You also should not paste all 36 files into the system prompt — module 05 already measured what that costs.

**Retrieval** is the boring, correct move: turn the question into a vector, find the nearest files, put *those* files in the prompt, then ask. That is single-shot RAG. One retrieve, one generate. Module 12 will loop it. Module 14 will show that a retrieved file is still untrusted data.

```mermaid
graph TD
    A[36 files on disk] --> B[Chroma]

    C[Question] -->|Embed question| D[Question embedding]
    D -->|Find nearest k documents| B

    B --> E[Retrieved file content]
    E --> F[Prompt]
    C --> F

    F --> G[LLM]
    G --> H[Answer]
```


## 1. Learn

```
08/09  the fact is in Chinook — query it
02/03  the fact is in a CSV — call a tool
05     the list is a budget
10     the model wrote code
11     you are here — the fact is in a folder of markdown
12     retrieve, assess, retrieve again
14     a retrieved file can carry instructions
```

Three places an answer can live, and RAG is only one of them:

| Question | Where the answer is | What to do |
|---|---|---|
| How many invoices does Helena have? | `chinook.db` | SQL. Module 08. |
| Sydney to Madrid? | `flight_data.csv` | A tool. Module 02. |
| How long do I have to return an unopened CD? | `data/corpus/` | Retrieve, then generate. |
| What is next year's revenue target? | Nowhere in this repo | Say you do not know. |

If you retrieve for Helena's invoice count you will get a policy about invoices, not the number 7. RAG is for unstructured text you cannot otherwise reach. It is not a default.

**Chunking.** These files are already short. Today **one file is one chunk**. That is a decision, not a library. Module 12 can split. We will not.

**The store.** The slides name four ways over one corpus. We run the second.

| Pass | What it is | In this notebook |
|---|---|---|
| From scratch | Cosine similarity in about ten lines | Not today. The idea is on the slide so a vector store does not look like magic. |
| A local vector store | Persistent, metadata filtering, laptop | **This.** Chroma, in this process. OpenAI embeds; Chroma only stores and ranks. |
| An indexing layer | Chunking and ingestion above the store | LlamaIndex is the name. We do not install it. One file is already one chunk. |
| A managed service | Same corpus, hosted | Instructor only, if at all, in module 16. |

The reasoning loop above the store is the durable thing. The store underneath it is swappable.

Two questions in this corpus have **no** answer. They are on purpose. An honest system says so. A chatbot with a vector store invents a mobile number.


## 2. Do

### Load the environment and read one file yourself


In [16]:
from pathlib import Path
import os

from chromadb import Client
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model = os.environ.get("MODEL_DEFAULT", "").strip()
embed_model = os.environ.get("EMBEDDING_MODEL", "").strip()
assert api_key, "OPENAI_API_KEY is missing."
assert model, "MODEL_DEFAULT is missing from .env."
assert embed_model, "EMBEDDING_MODEL is missing. Copy the line from .env.example."

client = OpenAI()
CORPUS = ROOT / "data" / "corpus"
files = sorted(CORPUS.glob("*.md"))
print("OPENAI_API_KEY is set:", True)
print("MODEL_DEFAULT:", model)
print("EMBEDDING_MODEL:", embed_model)
print("n files:", len(files))
print("first five:", [p.name for p in files[:5]])
print()
print("--- policy_returns.md ---")
print((CORPUS / "policy_returns.md").read_text())


OPENAI_API_KEY is set: True
MODEL_DEFAULT: gpt-5.4-nano
EMBEDDING_MODEL: text-embedding-3-small
n files: 36
first five: ['policy_accounts.md', 'policy_cancellations.md', 'policy_damaged_media.md', 'policy_data_retention.md', 'policy_digital_downloads.md']

--- policy_returns.md ---
# Returns

Physical items (CDs, vinyl, boxed sets) may be returned within 30 days of the invoice date if unopened.

Opened physical media can be returned within 14 days only if the disc is defective. A replacement is preferred over a refund.

Digital downloads and streamed albums cannot be returned. See policy_digital_downloads.md.

Start a return by writing to support. Include the invoice number. Support will reply within the SLA in policy_support_hours.md.



30 days if unopened. 14 if opened and defective. Digital cannot be returned. You did not need a model.

### An embedding is a list of floats

Every chunk of text you have sent to a model so far became tokens, then attention, then a next-token guess. An embedding stops one step earlier: the model reads the whole text once and hands back a fixed-length list of numbers — a point in space. Two texts with similar meaning land at nearby points, even with no words in common. Two unrelated texts land far apart. That is the whole idea. Everything else today is finding "nearby" fast.

Same API family, different endpoint: `embeddings.create`, not `chat.completions.create`. We print the length and the first eight numbers so you have seen the actual object once. The rest of the vector is more of the same — not something you are meant to read by eye.

In [17]:
sample = "How long do I have to return an unopened CD?"
emb = client.embeddings.create(model=embed_model, input=sample)
vec = emb.data[0].embedding
print("n dimensions:", len(vec))
print("first 8:     ", [round(x, 5) for x in vec[:8]])
print("prompt_tokens (this embed):", emb.usage.prompt_tokens)


n dimensions: 1536
first 8:      [0.00743, 0.07263, 0.01949, 0.0014, 0.01953, 0.00813, 0.008, 0.00896]
prompt_tokens (this embed): 12


That list of 1536 numbers has a name: a **vector**. That is the "vector" in "vector database" — nothing more mysterious than a list of floats, and Chroma's whole job is storing thousands of them and finding the nearest ones fast.

### How near is near? Two rulers, by hand

A vector on its own is not useful. What makes it useful is a way to measure how close two of them are — a ruler. Earlier modules built the agent loop by hand before reaching for a framework (module 03, then 08 and 09). Same move here: measure two questions yourself, so "nearest neighbour" stops being a phrase a library does for you and becomes ten lines of arithmetic you can read.

Two sentences to compare against `sample`, the CD-return question above:

- `near` — a paraphrase that shares **zero words** with `sample`: a sealed vinyl record, not an unopened CD.
- `far` — an unrelated sentence about the weather.

If embeddings work, `near` should measure close to `sample` and `far` should not, purely on meaning.

In [18]:
import math


def embed(text: str) -> list[float]:
    return client.embeddings.create(model=embed_model, input=text).data[0].embedding


def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    return dot / (norm_a * norm_b)


near = "What's the return window for a sealed vinyl record?"
far = "What's the weather like in Prague today?"

vec_near = embed(near)
vec_far = embed(far)

print("cosine similarity (bigger = more alike, 1.0 is identical direction)")
print("sample vs near:", round(cosine_similarity(vec, vec_near), 4))
print("sample vs far: ", round(cosine_similarity(vec, vec_far), 4))


cosine similarity (bigger = more alike, 1.0 is identical direction)
sample vs near: 0.6495
sample vs far:  0.0006


`near` shares no words with `sample` and still scored close to **1**. `far` shares no words either and scored close to **0**. Cosine similarity measures the angle between two points, not the words — this is the whole reason retrieval survives a paraphrase, a typo, or different wording for the same question.

### A second ruler: Euclidean distance

Cosine asks "do these two vectors point the same direction?" Euclidean distance asks a plainer question: "how far apart are these two points?" — the straight-line distance formula from school, just in 1536 dimensions instead of 2. Smaller is nearer, the opposite direction from cosine similarity.

Chroma will use exactly this second ruler in a few cells. Keep watching the numbers.

In [19]:
def squared_euclidean(a, b):
    return sum((x - y) ** 2 for x, y in zip(a, b))


print("squared Euclidean distance (smaller = closer, 0.0 is identical)")
print("sample vs near:", round(squared_euclidean(vec, vec_near), 4))
print("sample vs far: ", round(squared_euclidean(vec, vec_far), 4))


squared Euclidean distance (smaller = closer, 0.0 is identical)
sample vs near: 0.7007
sample vs far:  1.9991


Both rulers agree: `near` is close, `far` is not. They will not always agree on order for every pair — they are different formulas — but they agree here, and that is enough to trust either one.

Two sentences and one comparison is arithmetic you can do in your head. Thirty-six files and one question is 36 comparisons. Thirty-six thousand files is 36,000 comparisons, redone from scratch on every single question, in a `for` loop that gets slower as the corpus grows and remembers nothing between questions. A vector store's job is exactly that loop — indexed, so it does not check every file, and stored, so it does not re-embed anything it has already seen. That is the whole pitch for Chroma. Nothing above stops being true. It just stops being your problem.

One prediction before you run the next cells. Chroma will use **squared Euclidean distance**, the second ruler, not cosine. `policy_returns.md` is the file that answers `sample`. Compute its squared Euclidean distance to `sample` by hand, right now, before Chroma ever sees it:

In [20]:
returns_vec = embed((CORPUS / "policy_returns.md").read_text())

print("by hand:", round(squared_euclidean(vec, returns_vec), 4))
print("remember this number — Chroma will report the same one in a few cells")


by hand: 0.7721
remember this number — Chroma will report the same one in a few cells


That is the question, as far as the store is concerned. Nearness is a distance between two of those lists — exactly the arithmetic you just did by hand, run automatically and at scale.

### Load the folder

One id per file. The document is the file text. We embed each one ourselves and hand Chroma the vectors, so the store is not a second embedding library. `embed` below is the same helper you already defined — no new function, just the same one on 36 more inputs.

In [21]:
ids = []
documents = []
metadatas = []
vectors = []
for path in files:
    text = path.read_text()
    ids.append(path.name)
    documents.append(text)
    metadatas.append({"path": path.name})
    vectors.append(embed(text))

chroma = Client()
collection = chroma.get_or_create_collection("corpus")
collection.add(ids=ids, documents=documents, metadatas=metadatas, embeddings=vectors)
print("stored:", collection.count())


stored: 36


We used `Client()` — Chroma's **in-memory** client. All 36 vectors live in this Python process's RAM, and that memory is shared across the whole kernel, not just this variable: a second `Client()` call in the same session sees the same collection. That is why the cell above used `get_or_create_collection`, not `create_collection` — re-run it as many times as you like, kernel restarts aside, and it will not complain that "corpus" already exists.

Close the kernel and the collection is gone; the next run re-embeds the folder from scratch, exactly what just happened above.

The alternative is one constructor different: `PersistentClient(path="some/folder")` writes the same data to disk, so a second process — a restart, a different notebook, a running service — can open that path and query it without re-embedding anything.

| | `Client()` — in memory | `PersistentClient(path=...)` — on disk |
|---|---|---|
| Survives a kernel restart | No | Yes |
| Setup | Nothing to manage | A folder on disk |
| Good for | A notebook, a small corpus you rebuild often, a test | A real service, a corpus too large to re-embed on every start, more than one process reading the same store |

36 files re-embeds in a few seconds for a fraction of a cent, so in memory is the right call here — the same reasoning module 08 used for `SQLiteSession` in memory. A real corpus is bigger, embedding is not free, and a service should not have to wait for a re-embed on every restart. That is when `PersistentClient` earns its keep. We stay on `Client()` for the rest of this module; the only change to reach for the other one is the constructor.

### Retrieve, no generate

Nearest three files for the return-window question. Look at **names and distances** before anyone writes a sentence.

In [22]:
def retrieve(question: str, k: int = 3):
    q = embed(question)
    got = collection.query(query_embeddings=[q], n_results=k)
    rows = []
    for i in range(len(got["ids"][0])):
        rows.append(
            {
                "id": got["ids"][0][i],
                "distance": got["distances"][0][i],
                "document": got["documents"][0][i],
            }
        )
    return rows


QUESTION = "How long do I have to return an unopened CD?"
hits = retrieve(QUESTION)
for row in hits:
    print(f"{row['distance']:.3f}  {row['id']}")
    print(row["document"].splitlines()[0])
    print()


0.772  policy_returns.md
# Returns

1.071  policy_warranty.md
# Warranty on physical media

1.140  policy_damaged_media.md
# Damaged media



`policy_returns.md` should be first, at **0.772** — recognise that number? That is the squared Euclidean distance you predicted and computed by hand a few cells ago, matching to three decimal places. Chroma did not do anything you could not have done yourself; it just does it fast, indexed, and against 36 files instead of one.

Distance is smaller-is-nearer. If a ticket about a warped vinyl outranks the policy, say so — the question used "CD" and the ticket used "vinyl."

In [23]:
def answer(question: str, k: int = 3):
    hits = retrieve(question, k=k)
    packed = "\n\n".join(f"# {row['id']}\n{row['document']}" for row in hits)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer using only the documents. "
                    "If they do not contain the answer, say you do not know. "
                    "Do not invent a number, a date, or a phone number."
                ),
            },
            {
                "role": "user",
                "content": "Documents:\n\n" + packed + "\n\nQuestion: " + question,
            },
        ],
        max_completion_tokens=160,
        reasoning_effort="none",
    )
    return hits, response.choices[0].message.content, response.usage.prompt_tokens


hits, text, prompt_tokens = answer(QUESTION)
print("used:", [row["id"] for row in hits])
print("prompt_tokens:", prompt_tokens)
print()
print(text)


used: ['policy_returns.md', 'policy_warranty.md', 'policy_damaged_media.md']
prompt_tokens: 301

You may return an unopened physical item (including a CD) within **30 days of the invoice date**.


That sentence should be the 30-day rule, from the file you already read.

### A question the folder cannot answer

Same function. The corpus has no 2014 plan and no executive's personal number.


In [24]:
UNANSWERABLE = "What is the CEO's personal mobile number?"
hits_u, text_u, tokens_u = answer(UNANSWERABLE)
print("used:", [row["id"] for row in hits_u])
print("prompt_tokens:", tokens_u)
print()
print(text_u)


used: ['policy_escalations.md', 'policy_support_reps.md', 'policy_privacy.md']
prompt_tokens: 306

I don’t know. The provided documents only mention that the shop does not publish the support manager’s personal phone number, and they do not include any CEO personal mobile number.


If it said it does not know, retrieval did its job and the prompt held. If it invented a dollar figure, you have a chatbot that happened to search first. Module 12 is one response to that. "I do not know" is a valid outcome. It is not a broken cell.

## 3. Observe

Five questions. Retrieve only — no generate. Print the top file. You should be able to predict two misses before the cell runs.


In [25]:
questions = [
    "How long do I have to return an unopened CD?",
    "When is support staffed?",
    "Do gift cards expire?",
    "What is Chinook's revenue target for 2014?",
    "What is the CEO's personal mobile number?",
]
print(f"{'top file':<36} {'dist':>6}  question")
for q in questions:
    row = retrieve(q, k=1)[0]
    print(f"{row['id']:<36} {row['distance']:6.3f}  {q}")


top file                               dist  question
policy_returns.md                     0.772  How long do I have to return an unopened CD?
policy_support_reps.md                0.943  When is support staffed?
policy_gift_cards.md                  0.584  Do gift cards expire?
ticket_04_canada_free_shipping.md     1.498  What is Chinook's revenue target for 2014?
policy_escalations.md                 1.394  What is the CEO's personal mobile number?


The last two will still return *a* file. Nearest is not the same as relevant. The generate step is what has to refuse.

Things to notice:

- The embedding is the only new API. Chroma did not call OpenAI.
- `prompt_tokens` on the generate is the three files plus the question, not the whole folder. That is module 05, applied to retrieval.
- A top hit on an unanswerable question is not a bug in Chroma. It is why "always answer from context" is an incomplete instruction.

LlamaIndex would wrap `embed` + `add` + `query` in an `Index`. Pinecone would put the same vectors on someone else's machine. The loop above them does not change. We stay on Chroma.

## 4. Challenge

Same `answer` function (or the same two steps written out). A new question:

> What is the student discount on physical items?

Bind:

- `hits` — the retrieve rows
- `text` — the generated sentence

The next cell checks that a sentence came back and that it mentions **10**. It does not score the wording. We will look at `policy_student_discount.md` in the debrief.


In [ ]:
# hits, text = ...


In [ ]:
assert hits and len(hits) >= 1, "hits should be the retrieve rows"
assert text and str(text).strip(), "text should be the generated sentence"
assert "10" in str(text), "the policy is 10 percent off physical items"
print("looks good")
